<a href="https://colab.research.google.com/github/Gisellesg0/engenharia_de_promptIA/blob/main/aula08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Consulta de Endereço com viaCEP

Este script permite consultar informações de endereço utilizando o CEP através da API pública do viaCEP, com validação aprimorada.

In [33]:
import requests

def consultar_cep(cep):
    """
    Consulta a API do viaCEP para obter informações de endereço.

    Args:
        cep (str): O CEP a ser consultado (pode conter caracteres não numéricos).

    Returns:
        dict: Um dicionário com os dados do endereço se a consulta for bem-sucedida.
        str: Uma string indicando o tipo de erro:
             - "FORMAT_ERROR": se o CEP for malformado (não tiver 8 dígitos numéricos após limpeza).
             - "API_NOT_FOUND": se o CEP tiver formato válido, mas não for encontrado pela API do viaCEP.
        None: Em caso de outros erros de conexão/requisição ou exceções inesperadas.
    """
    # Guarda o CEP original para mensagens de erro
    original_cep = cep
    # Remove quaisquer caracteres não numéricos do CEP
    cep_limpo = ''.join(filter(str.isdigit, cep))

    if len(cep_limpo) != 8:
        # Este é um erro de formato local, antes da consulta à API.
        # A mensagem de erro será tratada pelo código que chama esta função.
        return "FORMAT_ERROR"

    url = f"https://viacep.com.br/ws/{cep_limpo}/json/"
    print(f"Consultando CEP: {cep_limpo}...")
    try:
        response = requests.get(url)
        response.raise_for_status()  # Levanta um HTTPError para respostas de status ruins (4xx ou 5xx)
        data = response.json()

        if 'erro' in data and data['erro']:
            # Isso indica que o CEP não foi encontrado pela API.
            return "API_NOT_FOUND"
        return data
    except requests.exceptions.HTTPError as http_err:
        print(f"Erro HTTP ao consultar CEP {cep_limpo}: {http_err}")
    except requests.exceptions.ConnectionError as conn_err:
        print(f"Erro de conexão ao consultar CEP {cep_limpo}: {conn_err}")
    except requests.exceptions.Timeout as timeout_err:
        print(f"Tempo limite excedido ao consultar CEP {cep_limpo}: {timeout_err}")
    except requests.exceptions.RequestException as req_err:
        print(f"Erro geral na requisição ao consultar CEP {cep_limpo}: {req_err}")
    except Exception as e:
        print(f"Ocorreu um erro inesperado: {e}")
    return None # Erro geral ou exceção inesperada

def formatar_endereco(dados_cep):
    """
    Formata os dados de endereço retornados pela API do viaCEP de maneira legível.

    Args:
        dados_cep (dict): Dicionário contendo os dados do endereço.

    Returns:
        str: Uma string formatada com as informações do endereço.
    """
    if not dados_cep:
        return "Nenhum dado de endereço para formatar."

    endereco_formatado = [
        f"CEP: {dados_cep.get('cep', 'N/A')}",
        f"Logradouro: {dados_cep.get('logradouro', 'N/A')}",
        f"Complemento: {dados_cep.get('complemento', 'N/A')}",
        f"Bairro: {dados_cep.get('bairro', 'N/A')}",
        f"Localidade: {dados_cep.get('localidade', 'N/A')}/{dados_cep.get('uf', 'N/A')}",
        f"IBGE: {dados_cep.get('ibge', 'N/A')}",
        f"GIA: {dados_cep.get('gia', 'N/A')}",
        f"DDD: {dados_cep.get('ddd', 'N/A')}",
        f"SIAFI: {dados_cep.get('siafi', 'N/A')}"
    ]
    return "\n".join(endereco_formatado)

### Exemplo de Uso

Basta chamar a função `consultar_cep` com o CEP desejado. O código de exemplo abaixo demonstrará os diferentes retornos e mensagens de status.

In [34]:
# Exemplo de uso com um CEP válido
meu_cep_valido = "01001-000" # CEP da Sé, São Paulo
resultado_cep_valido = consultar_cep(meu_cep_valido)

print("\n--- Resultado para CEP Válido ---")
if isinstance(resultado_cep_valido, dict):
    print("Status: Encontrado e Válido.")
    print(formatar_endereco(resultado_cep_valido))
elif resultado_cep_valido == "FORMAT_ERROR":
    print(f"Status: Erro de Formato Local. O CEP '{meu_cep_valido}' não pôde ser processado.")
elif resultado_cep_valido == "API_NOT_FOUND":
    print(f"Status: CEP INCORRETO. O CEP '{meu_cep_valido}' não foi encontrado na API do viaCEP.")
else: # None para outros erros (ex: conexão, timeout)
    print(f"Status: Erro de Consulta. Não foi possível obter informações para o CEP '{meu_cep_valido}'.")

print("\n" + "="*50 + "\n")

# Exemplo de uso com um CEP que não existe na API (mas com formato válido)
meu_cep_nao_encontrado_api = "99999-999" # CEP que provavelmente não existe
resultado_cep_nao_encontrado = consultar_cep(meu_cep_nao_encontrado_api)

print("\n--- Resultado para CEP Não Encontrado na API ---")
if isinstance(resultado_cep_nao_encontrado, dict):
    print("Status: Encontrado e Válido.")
    print(formatar_endereco(resultado_cep_nao_encontrado))
elif resultado_cep_nao_encontrado == "FORMAT_ERROR":
    print(f"Status: Erro de Formato Local. O CEP '{meu_cep_nao_encontrado_api}' não pôde ser processado.")
elif resultado_cep_nao_encontrado == "API_NOT_FOUND":
    print(f"Status: CEP INCORRETO. O CEP '{meu_cep_nao_encontrado_api}' não foi encontrado na API do viaCEP.")
else: # None para outros erros
    print(f"Status: Erro de Consulta. Não foi possível obter informações para o CEP '{meu_cep_nao_encontrado_api}'.")

print("\n" + "="*50 + "\n")

# Exemplo de uso com um CEP malformado (que será limpo pela função, mas resultará em formato inválido)
cep_malformado = "88034.000-00-ABC"
resultado_cep_malformado = consultar_cep(cep_malformado)

print("\n--- Resultado para CEP Malformado ---")
if isinstance(resultado_cep_malformado, dict):
    print("Status: Encontrado e Válido.")
    print(formatar_endereco(resultado_cep_malformado))
elif resultado_cep_malformado == "FORMAT_ERROR":
    print(f"Status: Erro de Formato Local. O CEP '{cep_malformado}' não pôde ser processado.")
elif resultado_cep_malformado == "API_NOT_FOUND":
    print(f"Status: CEP INCORRETO. O CEP '{cep_malformado}' não foi encontrado na API do viaCEP.")
else: # None para outros erros
    print(f"Status: Erro de Consulta. Não foi possível obter informações para o CEP '{cep_malformado}'.")

Consultando CEP: 01001000...

--- Resultado para CEP Válido ---
Status: Encontrado e Válido.
CEP: 01001-000
Logradouro: Praça da Sé
Complemento: lado ímpar
Bairro: Sé
Localidade: São Paulo/SP
IBGE: 3550308
GIA: 1004
DDD: 11
SIAFI: 7107


Consultando CEP: 99999999...

--- Resultado para CEP Não Encontrado na API ---
Status: CEP INCORRETO. O CEP '99999-999' não foi encontrado na API do viaCEP.



--- Resultado para CEP Malformado ---
Status: Erro de Formato Local. O CEP '88034.000-00-ABC' não pôde ser processado.


# File Organizer by Extension

This program will organize files in a specified directory by creating new subdirectories based on the file extensions (e.g., `.pdf`, `.jpg`) and moving the files into these respective folders.

In [31]:
import os
import shutil

def organize_files_by_extension(source_directory):
    """
    Organiza arquivos no 'source_directory' fornecido em subdiretórios
    com base em suas extensões de arquivo.

    Args:
        source_directory (str): O caminho para o diretório a ser organizado.
    """
    if not os.path.exists(source_directory):
        print(f"Erro: O diretório '{source_directory}' não existe.")
        return

    print(f"Organizando arquivos em: {source_directory}")

    for filename in os.listdir(source_directory):
        file_path = os.path.join(source_directory, filename)

        # Ignorar diretórios
        if os.path.isdir(file_path):
            continue

        # Obter a extensão do arquivo
        _, extension = os.path.splitext(filename)
        extension = extension.lower() # Converter para minúsculas para consistência

        if not extension: # Ignorar arquivos sem extensão (ex: 'Makefile')
            print(f"Ignorando '{filename}': Nenhuma extensão de arquivo encontrada.")
            continue

        # Criar nome do diretório de destino (ex: '.pdf' -> 'PDF_Files')
        # Remover o ponto inicial e adicionar '_Files'
        dest_folder_name = extension[1:].upper() + "_Files"
        dest_directory = os.path.join(source_directory, dest_folder_name)

        # Criar o diretório de destino se ele não existir
        if not os.path.exists(dest_directory):
            os.makedirs(dest_directory)
            print(f"Diretório criado: {dest_directory}")

        # Mover o arquivo
        try:
            shutil.move(file_path, dest_directory)
            print(f"Moveu '{filename}' para '{dest_folder_name}'")
        except Exception as e:
            print(f"Erro ao mover '{filename}': {e}")

    print("Organização de arquivos concluída!")

## How to use the function:

1.  **Specify your target directory:** Replace `'/content/my_unsorted_files'` with the actual path to the directory you want to organize.
2.  **Run the cell:** Execute the Python cell below.

**Important Considerations:**
*   **Permissions:** Ensure the Python environment has the necessary read/write permissions for the specified directory.
*   **Existing Folders:** If a folder with the same name as a generated extension folder already exists, the files will be moved into it.
*   **Backup:** It's always a good practice to back up your files before running any script that moves or modifies them.

In [10]:
# Exemplo de Uso:
# Primeiro, vamos criar alguns arquivos fictícios e um diretório para fins de demonstração
# Você pode pular esta parte se já tiver um diretório com arquivos para organizar

# Definir um diretório de teste
test_directory = '/content/my_unsorted_files'

# --- Início da correção: Limpar o diretório de teste antes de criar os arquivos ---
# Se o diretório de teste existir, remove-o completamente para garantir um estado limpo
import shutil
if os.path.exists(test_directory):
    shutil.rmtree(test_directory)
    print(f"Diretório de teste existente removido: {test_directory}")
# --- Fim da correção ---

# Criar o diretório de teste se ele não existir
if not os.path.exists(test_directory):
    os.makedirs(test_directory)
    print(f"Diretório de teste criado: {test_directory}")

# Criar alguns arquivos fictícios com diferentes extensões
open(os.path.join(test_directory, 'document1.pdf'), 'a').close()
open(os.path.join(test_directory, 'image1.jpg'), 'a').close()
open(os.path.join(test_directory, 'report.docx'), 'a').close()
open(os.path.join(test_directory, 'code.py'), 'a').close()
open(os.path.join(test_directory, 'another_image.png'), 'a').close()
open(os.path.join(test_directory, 'data.csv'), 'a').close()
open(os.path.join(test_directory, 'text_file.txt'), 'a').close()
open(os.path.join(test_directory, 'no_extension'), 'a').close() # Arquivo sem extensão

print("Arquivos fictícios criados:")
for f in os.listdir(test_directory):
    print(f"  - {f}")

print("\n--- Executando organização ---\n")

# Chamar a função para organizar os arquivos
organize_files_by_extension(test_directory)

print("\n--- Após a organização ---\n")
# Verificar o conteúdo do diretório de teste após a organização
print(f"Conteúdo de '{test_directory}' após a organização:")
for root, dirs, files in os.walk(test_directory):
    level = root.replace(test_directory, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')

Diretório de teste existente removido: /content/my_unsorted_files
Diretório de teste criado: /content/my_unsorted_files
Arquivos fictícios criados:
  - image1.jpg
  - no_extension
  - report.docx
  - text_file.txt
  - another_image.png
  - document1.pdf
  - code.py
  - data.csv

--- Executando organização ---

Organizando arquivos em: /content/my_unsorted_files
Diretório criado: /content/my_unsorted_files/JPG_Files
Moveu 'image1.jpg' para 'JPG_Files'
Ignorando 'no_extension': Nenhuma extensão de arquivo encontrada.
Diretório criado: /content/my_unsorted_files/DOCX_Files
Moveu 'report.docx' para 'DOCX_Files'
Diretório criado: /content/my_unsorted_files/TXT_Files
Moveu 'text_file.txt' para 'TXT_Files'
Diretório criado: /content/my_unsorted_files/PNG_Files
Moveu 'another_image.png' para 'PNG_Files'
Diretório criado: /content/my_unsorted_files/PDF_Files
Moveu 'document1.pdf' para 'PDF_Files'
Diretório criado: /content/my_unsorted_files/PY_Files
Moveu 'code.py' para 'PY_Files'
Diretório cri

# Nova seção

In [39]:
# Este script já foi desenvolvido e está disponível nas células anteriores.
# Abaixo está um exemplo de como utilizar as funções `consultar_cep` e `formatar_endereco`
# que já foram definidas no notebook, mas agora com a geração de exceções explícitas.

def obter_endereco_com_excecoes(cep):
    """
    Chama consultar_cep e levanta exceções específicas para CEPs inválidos.
    """
    resultado = consultar_cep(cep)
    if resultado == "FORMAT_ERROR":
        raise ValueError(f"CEP Inválido: O formato do CEP '{cep}' está incorreto. Verifique se tem 8 dígitos numéricos.")
    elif resultado == "API_NOT_FOUND":
        raise ValueError(f"CEP Não Encontrado: O CEP '{cep}' não foi localizado na base de dados do viaCEP.")
    elif resultado is None:
        raise ConnectionError(f"Erro de Conexão/Requisição: Não foi possível consultar o CEP '{cep}'.")
    return resultado

# --- Exemplo de Uso com Tratamento de Exceções ---

print("\n" + "="*50 + "\n")
print("### Exemplo com CEP Válido ###")
meu_cep_valido = "01001-000"
try:
    dados_endereco = obter_endereco_com_excecoes(meu_cep_valido)
    print(f"--- Consulta para o CEP: {meu_cep_valido} ---")
    print("Status: Encontrado e Válido.")
    print(formatar_endereco(dados_endereco))
except ValueError as e:
    print(f"Erro de Validação: {e}")
except ConnectionError as e:
    print(f"Erro de Conexão: {e}")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

print("\n" + "="*50 + "\n")
print("### Exemplo com CEP Não Encontrado na API ###")
meu_cep_nao_encontrado = "99999-999"
try:
    dados_endereco = obter_endereco_com_excecoes(meu_cep_nao_encontrado)
    print(f"--- Consulta para o CEP: {meu_cep_nao_encontrado} ---")
    print("Status: Encontrado e Válido.")
    print(formatar_endereco(dados_endereco))
except ValueError as e:
    print(f"Erro de Validação: {e}")
except ConnectionError as e:
    print(f"Erro de Conexão: {e}")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

print("\n" + "="*50 + "\n")
print("### Exemplo com CEP Malformatado ###")
meu_cep_malformatado = "88034.000-00-ABC"
try:
    dados_endereco = obter_endereco_com_excecoes(meu_cep_malformatado)
    print(f"--- Consulta para o CEP: {meu_cep_malformatado} ---")
    print("Status: Encontrado e Válido.")
    print(formatar_endereco(dados_endereco))
except ValueError as e:
    print(f"Erro de Validação: {e}")
except ConnectionError as e:
    print(f"Erro de Conexão: {e}")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")




### Exemplo com CEP Válido ###
Consultando CEP: 01001000...
--- Consulta para o CEP: 01001-000 ---
Status: Encontrado e Válido.
CEP: 01001-000
Logradouro: Praça da Sé
Complemento: lado ímpar
Bairro: Sé
Localidade: São Paulo/SP
IBGE: 3550308
GIA: 1004
DDD: 11
SIAFI: 7107


### Exemplo com CEP Não Encontrado na API ###
Consultando CEP: 99999999...
Erro de Validação: CEP Não Encontrado: O CEP '99999-999' não foi localizado na base de dados do viaCEP.


### Exemplo com CEP Malformatado ###
Erro de Validação: CEP Inválido: O formato do CEP '88034.000-00-ABC' está incorreto. Verifique se tem 8 dígitos numéricos.


### Consulta Interativa de CEP

Use os campos abaixo para inserir um CEP e consultá-lo. O resultado será exibido abaixo.

In [40]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Criar um campo de texto para o CEP
cep_input = widgets.Text(
    value='',
    placeholder='Digite o CEP (ex: 01001-000)',
    description='CEP:',
    disabled=False
)

# Criar um botão para consultar o CEP
consultar_button = widgets.Button(
    description='Consultar CEP',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Clique para consultar o CEP',
    icon='search' # (FontAwesome names without the `fa-` prefix)
)

# Área de saída para exibir os resultados
output_area = widgets.Output()

def on_button_click(b):
    with output_area:
        output_area.clear_output()
        cep = cep_input.value
        print(f"Consultando CEP: {cep}...")
        try:
            dados_endereco = obter_endereco_com_excecoes(cep)
            print(f"--- Resultado para o CEP: {cep} ---")
            print("Status: Encontrado e Válido.")
            print(formatar_endereco(dados_endereco))
        except ValueError as e:
            print(f"Erro de Validação: {e}")
        except ConnectionError as e:
            print(f"Erro de Conexão: {e}")
        except Exception as e:
            print(f"Ocorreu um erro inesperado: {e}")

# Associar a função ao evento de clique do botão
consultar_button.on_click(on_button_click)

# Exibir os widgets
display(cep_input, consultar_button, output_area)

Text(value='', description='CEP:', placeholder='Digite o CEP (ex: 01001-000)')

Button(button_style='info', description='Consultar CEP', icon='search', style=ButtonStyle(), tooltip='Clique p…

Output()